# Implementacion Algoritmo - Programacion dinamica (memorization)

In [ ]:
import time

In [ ]:
class Tarea:
    def __init__(self, id: str, tiempo: int, prioridad: int, dependencias: list[str]):
        self.id = id
        self.tiempo = tiempo
        self.prioridad = prioridad
        self.dependencias = dependencias

    def __repr__(self):
        return str(self.id)

### Complejidades

- #### Metodo __init__
    - complejidad temporal: O(1)
    - complejidad espacial: O(1)
- #### Metodo __repr__
    - complejidad temporal: O(1)
    - complejidad espacial: O(1)

In [ ]:
def dp_servidores_memo(tareas: list[Tarea], num_servidores: int):
    total_tareas = len(tareas)
    memo = {}

    def resolver(completadas: set, fin_servidor: tuple, fin_tarea: dict):
        estado_servidores = tuple(sorted(fin_servidor))
        estado_key = (frozenset(completadas), estado_servidores)

        if estado_key in memo:
            return memo[estado_key]

        if len(completadas) == total_tareas:
            tiempo_max = max(fin_servidor, default=0)
            return (tiempo_max, 0, [], [])

        disponibles = [
            t for t in tareas 
            if t.id not in completadas and all(dep in completadas for dep in t.dependencias)
        ]

        mejor_tiempo = float("inf")
        mejor_costo = float("inf")
        mejor_orden = []
        mejor_asignaciones = []

        for tarea in disponibles:
            fin_deps = max((fin_tarea[dep] for dep in tarea.dependencias), default=0)

            for s in range(num_servidores):
                inicio = max(fin_servidor[s], fin_deps)
                fin = inicio + tarea.tiempo

                nuevos_servidores = list(fin_servidor)
                nuevos_servidores[s] = fin
                completadas.add(tarea.id)
                fin_tarea[tarea.id] = fin

                t_sub, c_sub, ord_sub, asig_sub = resolver(
                    completadas, tuple(nuevos_servidores), fin_tarea
                )

                costo_actual = tarea.prioridad * fin + c_sub

                if t_sub < mejor_tiempo or (t_sub == mejor_tiempo and costo_actual < mejor_costo):
                    mejor_tiempo = t_sub
                    mejor_costo = costo_actual
                    mejor_orden = [tarea] + ord_sub
                    mejor_asignaciones = [s] + asig_sub

                completadas.remove(tarea.id)
                del fin_tarea[tarea.id]

        resultado = (mejor_tiempo, mejor_costo, mejor_orden, mejor_asignaciones)
        memo[estado_key] = resultado
        return resultado

    tiempo_total, costo_total, orden, asignacion = resolver(
        completadas=set(), 
        fin_servidor=tuple([0] * num_servidores), 
        fin_tarea={}
    )

    return (tiempo_total, orden, tuple(asignacion))

### Complejidades

- #### Método "dp_servidores_memo"
    - complejidad temporal: O(s · n · m (n + m log m))
    - complejidad espacial: O(s · (n + m))

- #### Método "resolver"
    - complejidad temporal: O(s · n · m (n + m log m))
    - complejidad espacial: O(n² + m)